In [2]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.data.load_data import load_resumes, load_jobs
from src.data.preprocess import clean_text
from src.models.recommender import build_recommender, recommend_jobs

In [3]:
resumes = load_resumes()
jobs = load_jobs().sample(50000, random_state=42).reset_index(drop=True)
jobs['clean_text'] = jobs['description'].apply(clean_text)
print("✅ Jobs ready:", jobs.shape)
print("✅ Resumes ready:", resumes.shape)

✅ Total jobs in merged corpus: 148771
✅ Columns: ['title', 'company', 'location', 'skills', 'description', 'experience']
✅ Jobs ready: (50000, 7)
✅ Resumes ready: (962, 2)


In [4]:
print("Building TF-IDF matrix... (may take 2-3 mins)")
vectorizer, job_matrix = build_recommender(jobs, text_col='clean_text')
print("✅ Recommender built!")
print("Job matrix shape:", job_matrix.shape)

Building TF-IDF matrix... (may take 2-3 mins)
✅ Recommender built!
Job matrix shape: (50000, 5000)


In [5]:
sample_resume = resumes['Resume'].iloc[0]
clean = clean_text(sample_resume)
actual_category = resumes['Category'].iloc[0]

print(f"Testing for category: {actual_category}")
print("-" * 50)

results = recommend_jobs(clean, jobs, vectorizer, job_matrix, top_n=10)
print(results.to_string())

Testing for category: Data Science
--------------------------------------------------
                                                         title                            company                 location                                                                                                                                                       skills  match_score
15268                                       Palantir Developer          Tata Consultancy Services            Cleveland, OH                                                                                                                                    Data Modelling , Big Data        35.93
44968                       Data Scientist, Advanced Analytics                     Analytics & BI                  Gurgaon   data science| machine learning| python| algorithms| java| data visualization| advanced analytics| data analysis| data mining| Eta| R| Network Optimization        34.55
30812                         Data 

In [6]:
sample_resume2 = resumes['Resume'].iloc[100]
clean2 = clean_text(sample_resume2)
category2 = resumes['Category'].iloc[100]

print(f"Testing for category: {category2}")
print("-" * 50)
results2 = recommend_jobs(clean2, jobs, vectorizer, job_matrix, top_n=10)
print(results2[['title', 'company', 'match_score']].to_string())

Testing for category: Advocate
--------------------------------------------------
                                                                          title                                             company  match_score
36971                  Program Director of Strategic Initiatives - Liberal Arts                The University of Texas at Arlington        25.09
462                                           Open Rank Lecturer, Communication                              University of Kentucky        24.47
19315                                                Human Resources Generalist                        Concordia University Chicago        22.55
30142                                                 Director, Student Affairs  Charles R. Drew University of Medicine and Science        22.46
6267   Adjunct Faculty, Information Technology, Software Design and Programming                                University of Denver        22.34
43983                                           

In [7]:
for idx in [0, 100, 200, 300]:
    resume_text = clean_text(resumes['Resume'].iloc[idx])
    cat = resumes['Category'].iloc[idx]
    res = recommend_jobs(resume_text, jobs, vectorizer, job_matrix, top_n=3)
    print(f"\n📄 Category: {cat}")
    print(res[['title', 'match_score']].to_string())
    print("-" * 50)


📄 Category: Data Science
                                     title  match_score
15268                   Palantir Developer        35.93
44968   Data Scientist, Advanced Analytics        34.55
30812     Data Protection Cognos Developer        34.27
--------------------------------------------------

📄 Category: Advocate
                                                          title  match_score
36971  Program Director of Strategic Initiatives - Liberal Arts        25.09
462                           Open Rank Lecturer, Communication        24.47
19315                                Human Resources Generalist        22.55
--------------------------------------------------

📄 Category: Mechanical Engineer
                                 title  match_score
12968  Mechanical Engineering Designer        37.03
19817       Mechanical Design Engineer        36.55
47330        Aerospace Project Manager        36.52
--------------------------------------------------

📄 Category: Civil Enginee

In [8]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
joblib.dump(job_matrix, '../models/job_matrix.pkl')
joblib.dump(jobs, '../models/jobs_df.pkl')
print("✅ Vectorizer, job matrix and jobs saved!")

✅ Vectorizer, job matrix and jobs saved!


In [9]:
# === Precision@K Evaluation ===
def precision_at_k(recommended_titles, relevant_keyword, k=10):
    top_k = recommended_titles[:k].tolist()
    hits = sum(1 for title in top_k 
               if relevant_keyword.lower() in str(title).lower())
    score = hits / k
    print(f"Precision@{k}: {score:.4f} ({hits}/{k} relevant jobs)")
    return score

# Test for different resume categories
print("=== Precision@K Evaluation ===\n")

test_cases = [
    (0,  "Data Science"),
    (50, "Java Developer"),
    (100, "HR"),
    (200, "Web Designing"),
]

for idx, category in test_cases:
    resume_text = clean_text(resumes['Resume'].iloc[idx])
    res = recommend_jobs(resume_text, jobs, vectorizer, job_matrix, top_n=10)
    keyword = category.split()[0]
    print(f"Category: {category}")
    precision_at_k(res['title'], keyword, k=10)
    print()

=== Precision@K Evaluation ===

Category: Data Science
Precision@10: 0.6000 (6/10 relevant jobs)

Category: Java Developer
Precision@10: 0.3000 (3/10 relevant jobs)

Category: HR
Precision@10: 0.0000 (0/10 relevant jobs)

Category: Web Designing
Precision@10: 0.0000 (0/10 relevant jobs)

